# 01 -- Baseline trade audit

Paired script: `analysis/analyse_baseline.py`. Per the reproducibility contract's rule 1
("every notebook has a paired `.py` pipeline containing the actual logic"), all the real
logic lives in that script and its shared modules (`metrics.py`, `trade_math.py`) -- this
notebook only builds a fixture and calls into it.

**Uses clearly-labelled SYNTHETIC fixture data.** Neither baseline EA (V6.37/V8.11) has a
committed real trade export -- `01_BASELINE/` contains only source code, screenshots, and
set files (see `TASK-028_PYTHON_STATISTICAL_LAB.md`'s Risks section). Per reproducibility
rule 7, the real-data run is marked PENDING at the end of this notebook rather than
fabricated.

In [1]:
import sys
import tempfile
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from analysis.analyse_baseline import run

In [2]:
# Synthetic fixture: 4 trades, hand-computable win rate / expectancy / profit
# factor / max drawdown (same numbers verified by hand in
# tests/test_analyse_baseline.py::test_summary_hand_computed).
tmp_dir = Path(tempfile.mkdtemp(prefix="themba_baseline_demo_"))
trades_csv = tmp_dir / "trades.csv"

pd.DataFrame(
    [
        {
            "trade_id": "t1",
            "symbol": "XAUUSD",
            "is_long": "True",
            "entry_time": "2026-07-21T00:00:00Z",
            "exit_time": "2026-07-21T01:00:00Z",
            "entry_price": 100.0,
            "exit_price": 102.0,
            "stop_price": 98.0,
            "profit": 40.0,
        },
        {
            "trade_id": "t2",
            "symbol": "XAUUSD",
            "is_long": "True",
            "entry_time": "2026-07-21T02:00:00Z",
            "exit_time": "2026-07-21T03:00:00Z",
            "entry_price": 100.0,
            "exit_price": 99.0,
            "stop_price": 98.0,
            "profit": -20.0,
        },
        {
            "trade_id": "t3",
            "symbol": "XAUUSD",
            "is_long": "False",
            "entry_time": "2026-07-21T04:00:00Z",
            "exit_time": "2026-07-21T05:00:00Z",
            "entry_price": 100.0,
            "exit_price": 95.0,
            "stop_price": 102.0,
            "profit": 50.0,
        },
        {
            "trade_id": "t4",
            "symbol": "XAUUSD",
            "is_long": "True",
            "entry_time": "2026-07-21T06:00:00Z",
            "exit_time": "2026-07-21T07:00:00Z",
            "entry_price": 100.0,
            "exit_price": 98.0,
            "stop_price": 98.0,
            "profit": -20.0,
        },
    ]
).to_csv(trades_csv, index=False)
# **Fixed, 2026-07-28 (Codex round-9 P2 finding 23): this previously printed
# the full absolute path, embedding a machine/session-specific temp
# directory (e.g. "C:\Users\THEMBA~1\AppData\Local\Temp\themba_baseline_demo_xxxxxxxx\...")
# into the notebook's own COMMITTED output -- not portable evidence, since
# the exact directory name is unique to whichever machine/session executed
# it. Only the filename (deterministic, portable) is printed now.
print(f"Synthetic trades written to a temp directory (filename: {trades_csv.name})")

Synthetic trades written to a temp directory (filename: trades.csv)


In [3]:
summary = run(
    trades_csv,
    starting_balance=1000.0,
    output_json=tmp_dir / "summary.json",
    repo_path=PROJECT_ROOT.parents[1],
)

print(f"n_trades                  = {summary['n_trades']}")
print(
    f"win_rate                  = {summary['win_rate']['value']:.4f} (95% CI [{summary['win_rate']['ci_lower']:.4f}, {summary['win_rate']['ci_upper']:.4f}])"
)
print(f"expectancy ($)            = {summary['expectancy_dollars']['value']:.2f}")
print(f"expectancy (R)            = {summary['expectancy_r']['value']:.4f}")
print(f"profit_factor             = {summary['profit_factor']:.4f}")
print(f"max_balance_drawdown_pct  = {summary['max_balance_drawdown_pct']:.4f}")
print(f"final_balance             = {summary['final_balance']:.2f}")

assert summary["n_trades"] == 4
assert abs(summary["win_rate"]["value"] - 0.5) < 1e-9
assert abs(summary["expectancy_dollars"]["value"] - 12.5) < 1e-9
assert abs(summary["profit_factor"] - 2.25) < 1e-9
# balance_curve = [1000, 1040, 1020, 1070, 1050] -- largest abs/pct decline
# both at peak=1040 -> trough=1020: 20, ~1.923% (hand-traced in
# tests/test_analyse_baseline.py::test_summary_hand_computed).
assert abs(summary["max_balance_drawdown_abs"] - 20.0) < 1e-9
assert abs(summary["max_balance_drawdown_pct"] - (20.0 / 1040.0)) < 1e-9
assert abs(summary["final_balance"] - 1050.0) < 1e-9

# **Added, 2026-07-22 Codex review finding (fifth round, R5F17): this
# notebook previously never displayed or hand-checked the newer summary
# fields (recovery_factor, balance_peak_giveback, longest_losing_streak,
# avg_winner/loser $ and n, avg_trade_duration_minutes and n,
# trades_per_day and its denominator) -- successful execution proved the
# cells ran, not that these newer contracts are correct. Every value
# below is hand-traced from the same 4-trade fixture above; see this
# cell's own balance_curve comment for the underlying balance steps.**
print(f"recovery_factor           = {summary['recovery_factor']:.4f}")
print(
    f"balance_peak_giveback     = armed={summary['balance_peak_giveback']['armed']} "
    f"n_trigger_events={summary['balance_peak_giveback']['n_trigger_events']} "
    f"trigger_indices={summary['balance_peak_giveback']['trigger_indices']} "
    f"max_giveback_pct={summary['balance_peak_giveback']['max_giveback_pct']:.6f}"
)
print(f"longest_losing_balance_step_streak = {summary['longest_losing_balance_step_streak']}")
print(
    f"avg_winner_dollars        = {summary['avg_winner_dollars']:.2f} "
    f"(n={summary['avg_winner_dollars_n']}, 95% CI "
    f"[{summary['avg_winner_dollars_ci_lower']:.4f}, {summary['avg_winner_dollars_ci_upper']:.4f}])"
)
print(
    f"avg_loser_dollars         = {summary['avg_loser_dollars']:.2f} "
    f"(n={summary['avg_loser_dollars_n']}, 95% CI "
    f"[{summary['avg_loser_dollars_ci_lower']:.4f}, {summary['avg_loser_dollars_ci_upper']:.4f}])"
)
print(
    f"avg_trade_duration_minutes= {summary['avg_trade_duration_minutes']:.2f} "
    f"(n={summary['avg_trade_duration_minutes_n']}, 95% CI "
    f"[{summary['avg_trade_duration_minutes_ci_lower']:.4f}, "
    f"{summary['avg_trade_duration_minutes_ci_upper']:.4f}])"
)
print(
    f"trades_per_day            = {summary['trades_per_day']:.6f} "
    f"(denominator: {summary['trades_per_day_denominator_days']:.6f} days, "
    f"source={summary['trades_per_day_denominator_source']!r})"
)

# recovery_factor = net_profit (1050 - 1000 = 50) / max_balance_drawdown_abs
# (20, already hand-verified above) = 2.5.
assert abs(summary["recovery_factor"] - 2.5) < 1e-9

# balance_peak_giveback, hand-traced over balance_curve =
# [1000, 1040, 1020, 1070, 1050] with the default arm_percent=1.0,
# floor_percent=0.5:
#   i=0 value=1000 peak=1000            -- not armed yet (0% >= 1%? no)
#   i=1 value=1040 peak=1040            -- arms (4% >= 1%); giveback 0%
#   i=2 value=1020 peak=1040 gb=20/1040=1.9231% -- triggers (event #1)
#   i=3 value=1070 peak=1070            -- new peak; giveback resets to 0%
#   i=4 value=1050 peak=1070 gb=20/1070=1.8692% -- triggers again (event #2)
# so 2 trigger events at indices [2, 4]; the worst (max) giveback is the
# first one, 20/1040, since 20/1070 < 20/1040.
assert summary["balance_peak_giveback"]["armed"] is True
assert summary["balance_peak_giveback"]["n_trigger_events"] == 2
assert summary["balance_peak_giveback"]["trigger_indices"] == [2, 4]
assert abs(summary["balance_peak_giveback"]["max_giveback_pct"] - (20.0 / 1040.0)) < 1e-9
assert summary["balance_peak_giveback"]["max_giveback_pct_index"] == 2

# **Renamed, 2026-07-22 Codex review finding (sixth round): this field
# previously named itself "longest_losing_streak", implying consecutive
# LOSING TRADES -- but it iterates one outcome per DISTINCT exit_time
# (balance steps), not one per trade. All 4 exit_times are distinct here
# (+40, -20, +50, -20), so the losses never repeat back to back and the
# longest run of consecutive losing BALANCE STEPS is 1 -- but see
# tests/test_analyse_baseline.py::test_simultaneous_losses_collapse_into_one_balance_step_streak
# for why this is NOT the same thing as "consecutive losing trades" once
# multiple trades share one exit_time.**
assert summary["longest_losing_balance_step_streak"] == 1

# winners = [40, 50] -> mean 45, n=2; losers = [-20, -20] -> mean -20, n=2.
# **Added, 2026-07-22 Codex review finding (sixth round): losers are
# IDENTICAL (-20, -20), a zero-variance sample, so its bootstrap CI
# collapses to the exact degenerate interval [-20, -20] -- not
# suppressed, per bootstrap_confidence_interval's own documented
# behavior for constant data (see test_sweep_ci_not_suppressed_for_
# constant_r_diffs in tests/test_parameter_stability.py for the same
# convention elsewhere in this project). winners genuinely vary (40,
# 50), so only its CI bounds are printed above, not hard-asserted here.**
assert abs(summary["avg_winner_dollars"] - 45.0) < 1e-9
assert summary["avg_winner_dollars_n"] == 2
assert abs(summary["avg_loser_dollars"] - (-20.0)) < 1e-9
assert summary["avg_loser_dollars_n"] == 2
assert abs(summary["avg_loser_dollars_ci_lower"] - (-20.0)) < 1e-9
assert abs(summary["avg_loser_dollars_ci_upper"] - (-20.0)) < 1e-9

# every trade in the fixture spans exactly 60 minutes (e.g. 00:00 -> 01:00)
# -- also a zero-variance sample, so its CI likewise collapses exactly.
assert abs(summary["avg_trade_duration_minutes"] - 60.0) < 1e-9
assert summary["avg_trade_duration_minutes_n"] == 4
assert abs(summary["avg_trade_duration_minutes_ci_lower"] - 60.0) < 1e-9
assert abs(summary["avg_trade_duration_minutes_ci_upper"] - 60.0) < 1e-9

# evaluation_period_days was NOT supplied to run() above, so trades_per_day
# falls back to the active trade envelope: first entry 00:00 -> last exit
# 07:00 = 7 hours = 7/24 days; 4 trades / (7/24) days = 96/7 trades/day.
assert summary["trades_per_day_denominator_source"] == "active_trade_envelope"
assert abs(summary["trades_per_day_denominator_days"] - (7.0 / 24.0)) < 1e-9
assert abs(summary["trades_per_day"] - (96.0 / 7.0)) < 1e-9

# **Added, 2026-07-22 Codex review finding (sixth round): this notebook
# previously only ever hand-checked the FALLBACK (active_trade_envelope)
# trades_per_day route -- never the AUTHENTICATED evaluation_period_days
# route the fifth round added. Re-run with a caller-supplied 30-day
# evaluation period (the same 4-trade fixture, unrelated to its own
# ~7-hour active envelope) to prove that route is genuinely exercised.**
summary_authenticated_period = run(
    trades_csv,
    starting_balance=1000.0,
    evaluation_period_days=30.0,
    repo_path=PROJECT_ROOT.parents[1],
)
print(
    f"trades_per_day (authenticated 30-day period) = "
    f"{summary_authenticated_period['trades_per_day']:.6f} "
    f"(source={summary_authenticated_period['trades_per_day_denominator_source']!r})"
)
assert (
    summary_authenticated_period["trades_per_day_denominator_source"]
    == "authenticated_evaluation_period"
)
assert abs(summary_authenticated_period["trades_per_day_denominator_days"] - 30.0) < 1e-9
assert abs(summary_authenticated_period["trades_per_day"] - (4.0 / 30.0)) < 1e-9

n_trades                  = 4
win_rate                  = 0.5000 (95% CI [0.1500, 0.8500])
expectancy ($)            = 12.50
expectancy (R)            = 0.5000
profit_factor             = 2.2500
max_balance_drawdown_pct  = 0.0192
final_balance             = 1050.00
recovery_factor           = 2.5000
balance_peak_giveback     = armed=True n_trigger_events=2 trigger_indices=[2, 4] max_giveback_pct=0.019231
longest_losing_balance_step_streak = 1
avg_winner_dollars        = 45.00 (n=2, 95% CI [40.0000, 50.0000])
avg_loser_dollars         = -20.00 (n=2, 95% CI [-20.0000, -20.0000])
avg_trade_duration_minutes= 60.00 (n=4, 95% CI [60.0000, 60.0000])
trades_per_day            = 13.714286 (denominator: 0.291667 days, source='active_trade_envelope')


trades_per_day (authenticated 30-day period) = 0.133333 (source='authenticated_evaluation_period')


## Real-data run: PENDING

No real baseline trade export exists yet. Once one is produced (via a future task bridging
a real MT5 statement export into `analyse_baseline.py`'s documented CSV schema), re-run this
notebook's second cell pointed at that file instead of the synthetic fixture.